In [1]:
from IPython import get_ipython
from IPython.display import display
import pandas as pd
import io

# Load datasets from the local file system
existing_df = pd.read_csv('val_data.csv')
manual_df = pd.read_csv('predicted_labels.csv')

# Preview the data222
print(existing_df.head())
print(manual_df.head())

     Id                                              Tweet  Hate  Fake
0  5709  still searching for nano gps chip in notes abh...     0     0
1  6668  dharam sirf ek hai wo sanatan dharm hai brbaki...     0     0
2  1035  amit shah ji kaun se jamaat mein gaye the arna...     1     1
3  2400  sir corona virus se hum jeet chuke hote agar y...     1     1
4  4170  har roz jakham kha ke hindus of bangladesh aur...     1     1
     Id                                              Tweet  Hate  Fake  \
0  5709  still searching for nano gps chip in notes abh...   NaN   NaN   
1  6668  dharam sirf ek hai wo sanatan dharm hai brbaki...   NaN   NaN   
2  1035  amit shah ji kaun se jamaat mein gaye the arna...   NaN   NaN   
3  2400  sir corona virus se hum jeet chuke hote agar y...   NaN   NaN   
4  4170  har roz jakham kha ke hindus of bangladesh aur...   NaN   NaN   

   Hate_pred  Fake_pred  
0          0          0  
1          0          0  
2          1          1  
3          1          1  

In [2]:
# Merge on Id
merged_df = pd.merge(existing_df, manual_df, on='Id', suffixes=('_existing', '_manual'))

# Show a few rows
merged_df.head()

,Id,Tweet_existing,Hate_existing,Fake_existing,Tweet_manual,Hate_manual,Fake_manual,Hate_pred,Fake_pred
0,5709,still searching for nano gps chip in notes abh...,0,0,still searching for nano gps chip in notes abh...,NaN,NaN,0,0
1,6668,dharam sirf ek hai wo sanatan dharm hai brbaki...,0,0,dharam sirf ek hai wo sanatan dharm hai brbaki...,NaN,NaN,0,0
2,1035,amit shah ji kaun se jamaat mein gaye the arna...,1,1,amit shah ji kaun se jamaat mein gaye the arna...,NaN,NaN,1,1
3,2400,sir corona virus se hum jeet chuke hote agar y...,1,1,sir corona virus se hum jeet chuke hote agar y...,NaN,NaN,1,1
4,4170,har roz jakham kha ke hindus of bangladesh aur...,1,1,har roz jakham kha ke hindus of bangladesh aur...,NaN,NaN,1,1


In [3]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, matthews_corrcoef

In [4]:
# Extract labels
hate_existing = merged_df['Hate_existing']
hate_manual = merged_df['Hate_pred']

fake_existing = merged_df['Fake_existing']
fake_manual = merged_df['Fake_pred']

In [5]:
print("Hate Label Comparison")
print("Accuracy:", accuracy_score(hate_manual, hate_existing))
print("Cohen’s Kappa:", cohen_kappa_score(hate_manual, hate_existing))
print("F1 Score (weighted):", f1_score(hate_manual, hate_existing, average='weighted'))
print("Matthews Corrcoef:", matthews_corrcoef(hate_manual, hate_existing))

Hate Label Comparison
Accuracy: 0.7847309136420526
Cohen’s Kappa: 0.5072640440575387
F1 Score (weighted): 0.791971344661937
Matthews Corrcoef: 0.516011003339783


In [6]:
print("\nFake Label Comparison")
print("Accuracy:", accuracy_score(fake_manual, fake_existing))
print("Cohen’s Kappa:", cohen_kappa_score(fake_manual, fake_existing))
print("F1 Score (weighted):", f1_score(fake_manual, fake_existing, average='weighted'))
print("Matthews Corrcoef:", matthews_corrcoef(fake_manual, fake_existing))


Fake Label Comparison
Accuracy: 0.7634543178973717
Cohen’s Kappa: 0.5199462124607799
F1 Score (weighted): 0.7669255833019061
Matthews Corrcoef: 0.5296452489302053


In [7]:
# Show samples where Hate labels differ
print("\nDisagreements in 'Hate':")
print(merged_df[merged_df['Hate_existing'] != merged_df['Hate_pred']][['Id', 'Tweet_existing', 'Hate_existing', 'Hate_pred']].head())

# Show samples where Fake/Faux labels differ
print("\nDisagreements in 'Fake':")
print(merged_df[merged_df['Fake_existing'] != merged_df['Fake_pred']][['Id', 'Tweet_existing', 'Fake_existing', 'Fake_pred']].head())


Disagreements in 'Hate':
      Id                                     Tweet_existing  Hate_existing  \
12  1502  kuch nahi vi ye govt ke sari galati ab jamaat ...              0   
13  1431  chalo humne man liya tablighi jamaat wale gala...              0   
15  4804  abe chutiye tune to pathaan ko b boycott kiya ...              1   
21  6771                      matt kar le maaje bache karva              1   
29  7770    yaha bhi koi tamil ya south ka nhi aana chahiye              1   

    Hate_pred  
12          1  
13          1  
15          0  
21          0  
29          0  

Disagreements in 'Fake':
      Id                                     Tweet_existing  Fake_existing  \
9   2707  small bt true wish jhoot bole corona pakre jo ...              0   
12  1502  kuch nahi vi ye govt ke sari galati ab jamaat ...              0   
13  1431  chalo humne man liya tablighi jamaat wale gala...              0   
19  3155  tu kya corona warrior hai lakh cr me se teko m...            

In [8]:
hate_existing = merged_df['Hate_existing']
hate_manual = merged_df['Hate_pred']
fake_existing = merged_df['Fake_existing']
fake_manual = merged_df['Fake_pred']

# Create dataframe for correlation analysis
correlation_df = pd.DataFrame({
    'Hate_existing': hate_existing,
    'Hate_pred': hate_manual,
    'Fake_existing': fake_existing,
    'Fake_pred': fake_manual
})

# Compute and display correlation matrix
correlation_matrix = correlation_df.corr()
print(correlation_matrix)

               Hate_existing  Hate_pred  Fake_existing  Fake_pred
Hate_existing       1.000000   0.516011       0.342134   0.486286
Hate_pred           0.516011   1.000000       0.439593   0.786315
Fake_existing       0.342134   0.439593       1.000000   0.529645
Fake_pred           0.486286   0.786315       0.529645   1.000000


In [9]:
from sklearn.metrics import accuracy_score

# Accuracy for 'Hate' label
hate_accuracy = accuracy_score(merged_df['Hate_pred'], merged_df['Hate_existing'])
print("Hate Label Accuracy:", round(hate_accuracy * 100, 2), "%")

# Accuracy for 'Fake' label
fake_accuracy = accuracy_score(merged_df['Fake_pred'], merged_df['Fake_existing'])
print("Fake Label Accuracy:", round(fake_accuracy * 100, 2), "%")

Hate Label Accuracy: 78.47 %
Fake Label Accuracy: 76.35 %


In [10]:
# Compare if both labels match in each row
both_match = (
    (merged_df['Hate_pred'] == merged_df['Hate_existing']) &
    (merged_df['Fake_pred'] == merged_df['Fake_existing'])
)

# Calculate overall accuracy
overall_accuracy = both_match.sum() / len(both_match)
print("Overall Label Agreement Accuracy:", round(overall_accuracy * 100, 2), "%")

Overall Label Agreement Accuracy: 62.95 %
